# Notebook19 — bounded whole JEPA profiling

Purpose: isolate full JEPA compute, compare the frozen18 learned kernel with a numerically equivalent implementation, and identify expensive operators. Inputs are discarded synthetic256pxRGB and registered model dimensions192/4/3. Output traces and summaries are under artifacts/notebook_19_validation/profile. Runtime normally2–5minutes, hard10minutes; no official pixels or scientific training. Rerun uses fresh discarded initialization; no long run occurs here. Profiling is explicitly separated from unprofiled timing.

In [ ]:
"""Bounded full JEPA profiling; synthetic inputs and no official pixels."""
from pathlib import Path
import copy,gc,gzip,hashlib,json,os,time,types,subprocess,threading
import torch,nbformat

def tagged(root,relative,name):
    path=Path(root)/relative;module=types.ModuleType(name);module.__file__=str(path)
    for cell in nbformat.read(path,4).cells:
        if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(path),'exec'),module.__dict__)
    return module

def operator_sources(root):
    result={}
    for relative in ('experiments/18_learned_kernel/operator.ipynb','experiments/19_compute_and_gate/operator.ipynb','experiments/19_compute_and_gate/profiling.ipynb'):
        path=Path(root)/relative
        if path.exists():
            code='\n'.join(c.source for c in nbformat.read(path,4).cells if c.cell_type=='code'and'ahr-export'in c.metadata.get('tags',[]))
            result[relative]=hashlib.sha256(code.encode()).hexdigest()
    return result

def profile_configuration():
    return {'kind':'vit','patch_size':8,'dim':192,'depth':4,'heads':3,'predictor_dim':128,'predictor_depth':2,'predictor_heads':4,
      'kernel15':{'variant':'optional_geometry','temperature_trainable':False,'spatial_length_trainable':False,'spatial_coefficient_trainable':False}}

def build_profile_model(root,variant,device):
    E=tagged(root,'experiments/15_vit_geometry/encoder.ipynb','profile19encoder')
    cfg=profile_configuration();torch.manual_seed(20262298)
    model=E.build_model(cfg,'vit'if variant=='vit'else'semantic')
    if variant!='vit':
        relative='experiments/18_learned_kernel/operator.ipynb'if variant=='original18'else'experiments/19_compute_and_gate/operator.ipynb'
        O=tagged(root,relative,'profile19'+variant)
        O.replace_encoder_mixers(model,{'rank':64,'feature_seed':180064,'local_radius':1,'spatial_mode':'disabled',
            'feature_temperature_min':.25,'feature_temperature_max':4.,'feature_temperature_initial':1.,'gate_initial':.1 if variant=='optimized19_gate01'else.5})
    return model.to(device).train()

def synthetic_profile_inputs(root,device,physical_batch=16,image_size=256):
    boot=tagged(root,'fmm_jepa/bootstrap.ipynb','profile19bootstrap');boot.install_notebook_package(root)
    from fmm_jepa.data import sample_masks
    generator=torch.Generator(device='cpu').manual_seed(20262298)
    images=torch.rand(physical_batch,3,image_size,image_size,generator=generator)
    masks=sample_masks((image_size//8,image_size//8),physical_batch,generator,target_count=2)
    return (images.to(device),masks['context_indices'].to(device),masks['context_valid'].to(device),
            [v.to(device)for v in masks['target_indices']],[v.to(device)for v in masks['target_valid']])

def whole_update(model,optimizer,inputs,accumulation=8):
    optimizer.zero_grad(set_to_none=True)
    for _ in range(accumulation):
        output=model(*inputs);(output['loss']/accumulation).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
    optimizer.step();model.update_teacher(.996)
    return float(output['loss'].detach())

def synchronize(device):
    if str(device).startswith('cuda'):torch.cuda.synchronize(device)


class ContinuousGPUObservation:
    def __init__(self,device):
        self.device=str(device);self.index=int(self.device.split(':')[-1]);self.samples=[];self.process=None;self.reader=None
    def start(self):
        self.process=subprocess.Popen(['nvidia-smi','pmon','-i',str(self.index),'-s','u','-d','1'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        def read():
            for line in self.process.stdout:
                line=line.strip()
                if not line or line.startswith('#'):continue
                fields=line.split()
                if len(fields)>=5:
                    def number(value):
                        try:return int(value)
                        except ValueError:return None
                    self.samples.append({'observed_unix':time.time(),'gpu':number(fields[0]),'pid':number(fields[1]),'type':fields[2],
                      'sm_percent':number(fields[3]),'memory_percent':number(fields[4]),'command':fields[-1],'raw':line})
        self.reader=threading.Thread(target=read,daemon=True);self.reader.start();time.sleep(1.1)
    def stop(self):
        if self.process:
            time.sleep(1.1);self.process.terminate()
            try:self.process.wait(timeout=3)
            except subprocess.TimeoutExpired:self.process.kill();self.process.wait(timeout=3)
        if self.reader:self.reader.join(timeout=3)

def _run_profile(root,artifact,variants=('vit','original18','optimized19'),device='cuda:1',warmup_updates=1,repeats=4,profile_updates=1,deadline=None,monitor=None):
    root=Path(root).resolve();artifact=Path(artifact).resolve();artifact.mkdir(parents=True,exist_ok=True)
    deadline=float(deadline or time.time()+600);torch.set_num_threads(2)
    if time.time()>=deadline:raise TimeoutError('Original bounded profiling deadline elapsed')
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    torch.use_deterministic_algorithms(True);inputs=synthetic_profile_inputs(root,device)
    rows=[]
    for variant in variants:
        if time.time()>=deadline:raise TimeoutError('Profiling cutoff; no performance conclusion')
        model=build_profile_model(root,variant,device);optimizer=torch.optim.AdamW([p for p in model.parameters()if p.requires_grad],lr=1e-4,weight_decay=.04)
        for _ in range(warmup_updates):whole_update(model,optimizer,inputs);synchronize(device)
        times=[];losses=[];windows=[]
        if str(device).startswith('cuda'):torch.cuda.reset_peak_memory_stats(device)
        baseline_allocated=torch.cuda.memory_allocated(device)if str(device).startswith('cuda')else None
        for _ in range(repeats):
            if time.time()>=deadline:raise TimeoutError('Profiling cutoff')
            synchronize(device);window_begin=time.time();started=time.perf_counter();loss=whole_update(model,optimizer,inputs);synchronize(device);window_end=time.time();windows.append([window_begin,window_end])
            times.append(time.perf_counter()-started);losses.append(loss)
        peak=torch.cuda.max_memory_allocated(device)if str(device).startswith('cuda')else None
        operators=[];trace=None;profiled_elapsed=None
        if profile_updates:
            activities=[torch.profiler.ProfilerActivity.CPU]
            if str(device).startswith('cuda'):activities.append(torch.profiler.ProfilerActivity.CUDA)
            synchronize(device);started=time.perf_counter()
            with torch.profiler.profile(activities=activities,record_shapes=True,profile_memory=True,with_stack=False)as profiler:
                for _ in range(profile_updates):whole_update(model,optimizer,inputs)
                synchronize(device)
            profiled_elapsed=time.perf_counter()-started
            events=sorted(profiler.key_averages(),key=lambda e:float(getattr(e,'self_device_time_total',0.)),reverse=True)
            for event in events[:40]:
                operators.append({'operator':event.key,'calls':event.count,'self_cpu_ms':event.self_cpu_time_total/1000.,
                  'self_device_ms':float(getattr(event,'self_device_time_total',0.))/1000.,'device_total_ms':float(getattr(event,'device_time_total',0.))/1000.,
                  'self_cpu_memory_bytes':event.self_cpu_memory_usage,'self_device_memory_bytes':getattr(event,'self_device_memory_usage',None)})
            raw=artifact/(variant+'-'+str(device).replace(':','_')+'-trace.json');profiler.export_chrome_trace(str(raw))
            compressed=raw.with_suffix('.json.gz')
            with raw.open('rb')as source,gzip.open(compressed,'wb')as target:
                import shutil;shutil.copyfileobj(source,target)
            raw.unlink();trace={'path':str(compressed),'sha256':hashlib.sha256(compressed.read_bytes()).hexdigest()}
        row={'variant':variant,'device':str(device),'status':'MEASURED','seconds_per_optimizer_update':times,
          'median_seconds_per_optimizer_update':float(torch.tensor(times).median()),'physical_batch':16,'accumulation':8,'effective_batch':128,
          'full_image_tokens':1024,'image_size':256,'context_tokens':int(inputs[1].shape[1]),'target_tokens':[int(v.shape[1])for v in inputs[3]],
          'peak_allocated_bytes':peak,'baseline_allocated_bytes':baseline_allocated,'losses':losses,'operators':operators,'trace':trace,
          'profiler_wall_seconds':profiled_elapsed,'unprofiled_measurement_windows_unix':windows,'measurement_scope':'Unprofiled full JEPA forward/backward, gradient clip, AdamW and EMA on preallocated synthetic inputs. Excludes real input decoding and checkpoint IO.',
          'profile_role':'Profiler perturbs timing; only unprofiled repeats estimate speed. Fresh discarded weights, repeated fixed synthetic masks; not an accuracy or complete training-throughput claim.'}
        rows.append(row);(artifact/'profile_progress.json').write_text(json.dumps({'status':'RUNNING','rows':rows,'sources':operator_sources(root),'official_test_pixels_read':0},indent=2)+'\n')
        del model
        del optimizer;gc.collect()
        if str(device).startswith('cuda'):torch.cuda.empty_cache()
    result={'status':'COMPLETE','rows':rows,'source_hashes':operator_sources(root),'deadline_unix':deadline,'official_test_pixels_read':0,
      'input_pipeline_workers':4,'input_pipeline_note':'Training pipeline uses4workers; compute-isolation timing above intentionally excludes data loading. Complete end-to-end training throughput must come from pilot timers.'}
    (artifact/'profile_result.json').write_text(json.dumps(result,indent=2)+'\n');return result


def run_profile(root,artifact,variants=('vit','original18','optimized19'),device='cuda:1',warmup_updates=1,repeats=4,profile_updates=1,deadline=None):
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    # A completed prior exploratory trace fixes the original phase clock.
    request_path=artifact/'profile_request.json';old_path=artifact/'profile_result.json'
    if request_path.exists():request=json.loads(request_path.read_text())
    else:
        inherited=json.loads(old_path.read_text()).get('deadline_unix')if old_path.exists()else None
        bound=float(inherited or deadline or time.time()+600)
        request={'started_unix':bound-600,'deadline_unix':bound,'maximum_seconds':600,'resume_policy':'No clock reset. Profiling uses discarded states; cutoff staysUNKNOWN.'}
        request_path.write_text(json.dumps(request,indent=2)+'\n')
    if deadline is not None and float(deadline)!=float(request['deadline_unix']):raise ValueError('Original profiling deadline changed')
    if time.time()>=request['deadline_unix']:raise TimeoutError('Original profiling phase has expired')
    if old_path.exists():
        old=json.loads(old_path.read_text())
        if not all('performance_claim_qualified'in row for row in old.get('rows',[])):
            for row in old.get('rows',[]):
                row['performance_claim_qualified']=False;row['performance_status']='UNKNOWN_NO_CONTINUOUS_GPU_OBSERVATION'
                row['qualification_scope']='Initial exploration had no continuous sampling; foreignPythonPID2702290 observed3%SM during this exploration, so no speedcomparison claim.'
            (artifact/'initial_exploration_unqualified.json').write_text(json.dumps(old,indent=2)+'\n')
    observation=ContinuousGPUObservation(device)if str(device).startswith('cuda')else None
    try:
        if observation:observation.start()
        result=_run_profile(root,artifact,variants,device,warmup_updates,repeats,profile_updates,request['deadline_unix'],observation)
    finally:
        if observation:
            observation.stop();(artifact/'gpu_observations.json').write_text(json.dumps({'own_pid':os.getpid(),'device':device,'samples':observation.samples},indent=2)+'\n')
    for row in result['rows']:
        if observation:
            windows=row['unprofiled_measurement_windows_unix'];low=min(w[0]for w in windows);high=max(w[1]for w in windows)
            samples=[s for s in observation.samples if low-1.2<=s['observed_unix']<=high+1.2]
            foreign=[s for s in samples if s['pid'] not in(None,os.getpid())and(s['sm_percent']or 0)>0]
            covered=bool(samples)and min(s['observed_unix']for s in observation.samples)<=low and max(s['observed_unix']for s in observation.samples)>=high
            row['gpu_observations']=samples;row['foreign_compute_observed']=foreign;row['performance_claim_qualified']=covered and not foreign
            row['performance_status']='MEASURED_EXCLUSIVE_OBSERVED'if row['performance_claim_qualified']else'UNKNOWN_FOREIGN_COMPUTE_OR_OBSERVATION_GAP'
            row['qualification_scope']='Continuous1Hzpmon covered the unprofiled windows; no positive foreignSM sample is accepted. Subsecond unsampled interference cannot be ruled out.'
        else:
            row['performance_claim_qualified']=True;row['performance_status']='MEASURED_CPU';row['qualification_scope']='CPU2threads; no GPUspeedclaim.'
    result['request']=request;result['status']='COMPLETE'if all(r['performance_claim_qualified']for r in result['rows'])else'UNKNOWN_PERFORMANCE_QUALIFICATION'
    result['source_hashes']=operator_sources(root);(artifact/'profile_result.json').write_text(json.dumps(result,indent=2)+'\n');return result


In [ ]:
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
if os.environ.get('FMM19_PROFILE_RUN','0')=='1':
    RESULT=run_profile(ROOT,ROOT/'artifacts/notebook_19_validation/profile',variants=tuple(os.environ.get('FMM19_PROFILE_VARIANTS','vit,original18,optimized19').split(',')),device=os.environ.get('FMM19_PROFILE_DEVICE','cuda:1'))
    print([(row['variant'],row['median_seconds_per_optimizer_update'],row['peak_allocated_bytes'])for row in RESULT['rows']])
else:
    print('Definitions ready. SetFMM19_PROFILE_RUN=1 for the bounded discarded synthetic profile.')